# GitHub Single Repo README Fetcher

이 노트북은 특정 GitHub 저장소 URL을 입력받아 해당 저장소의 `README.md` 내용을 가져오고 **로컬에 저장**합니다.

In [ ]:
import httpx
import re
import os
from pathlib import Path

def parse_github_url(url):
    """GitHub URL에서 owner와 repo 이름을 추출합니다."""
    pattern = r"github\.com/([^/]+)/([^/\.]+)"
    match = re.search(pattern, url)
    if match:
        return match.group(1), match.group(2)
    return None, None

def fetch_readme_raw(owner, repo):
    """GitHub의 raw content URL을 사용하여 README를 가져옵니다."""
    branches = ['main', 'master']
    
    for branch in branches:
        raw_url = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/README.md"
        response = httpx.get(raw_url)
        if response.status_code == 200:
            return response.text
    
    return None

def save_readme_single(owner, repo, content):
    """가져온 README를 파일로 저장합니다."""
    output_dir = Path("extracted_readmes")
    output_dir.mkdir(exist_ok=True)
    
    file_name = f"{owner}_{repo}_README.md"
    file_path = output_dir / file_name
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path

In [ ]:
# 사용법 예시
repo_url = "https://github.com/boostcampaitech8/pro-nlp-finalproject-nlp-01"

owner, repo = parse_github_url(repo_url)
if owner and repo:
    print(f"Target: {owner}/{repo}")
    readme_content = fetch_readme_raw(owner, repo)
    
    if readme_content:
        saved_path = save_readme_single(owner, repo, readme_content)
        print(f"성공! README가 저장되었습니다: {saved_path}")
        print("--- README Content (First 200 chars) ---")
        print(readme_content[:200])
    else:
        print("README를 찾을 수 없습니다.")
else:
    print("올바른 GitHub 저장소 URL을 입력해주세요.")